In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "PAQ_G.xpt"
csv_file = DATA_DIR / "PAQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 PAQ_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PAQ706",
    "PAQ605",
    "PAQ610",
    "PAD615",
    "PAQ620",
    "PAQ625",
    "PAD630",
    "PAQ635",
    "PAQ640",
    "PAD645",
    "PAQ650",
    "PAQ655",
    "PAD660",
    "PAQ665",
    "PAQ670",
    "PAD675",
    "PAD680",
    "PAQ710",
    "PAQ715",
    "PAAQUEX"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "PAQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9107:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 21:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,107 rows x 21 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Never globally replace this value before validating
# exactly where it occurs.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. VALIDATE EXACT TINY-VALUE PATTERN
# ============================================================

#
# Exact attached PAQ_G.xpt pattern:
#
# PAQ706 =    55
# PAD680 =     1
# PAQ710 = 1,175
# PAQ715 = 2,354
#
# TOTAL = 3,585
#
#
# CDC support:
#
# PAQ706:
#     valid range includes 0-7
#
# PAD680:
#     valid range includes 0-1380 minutes
#
# PAQ710:
#     0 = Less than 1 hour
#     CDC count = 1,175
#
# PAQ715:
#     0 = Less than 1 hour
#     CDC count = 2,354
#

EXPECTED_TINY_COUNTS = {
    "PAQ706": 55,
    "PAD680": 1,
    "PAQ710": 1175,
    "PAQ715": 2354
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated PAQ_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 3585:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact 3,585-value tiny-zero "
    "pattern validated."
)


# ============================================================
# 6. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Only restore these four validated fields.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 3585:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# numeric dataset should contain.
#
# ONLY validated tiny -> zero positions are changed.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining after correction:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 9. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# Always check BEFORE Int64 conversion.
#
# Attached PAQ_G contains NO genuine fractional fields.
#
# All time variables in this release are recorded as
# whole-number minutes/hours/categories.
#
# If fractional values unexpectedly appear:
#
#     STOP
#
# rather than rounding them.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in PAQ_G. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# PAQ_G contains no genuine decimal variables.
#
# Therefore all 21 fields can safely become
# pandas nullable Int64.
#
# NO .round() is used for conversion.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely become Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 21 PAQ_G variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 12. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches expected release."
    )


# ============================================================
# 13. CDC CHECK - PAQ706
# ============================================================

#
# Days physically active >=60 minutes.
#
# CDC:
#
# 0-7 = 2,299
# 99  = 1
# Missing = 6,807
#
# Attached XPT:
#
# 55 of the 0-7 values are zero and were
# decoded as TINY_VALUE.
#

s = df["PAQ706"]


print("\n--- PAQ706 ---")


print(
    "Restored zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            0,
            7
        ).sum()
    ) == 2299

    and int(
        s.eq(0).sum()
    ) == 55

    and int(
        s.eq(77).sum()
    ) == 0

    and int(
        s.eq(99).sum()
    ) == 1

    and int(
        s.isna().sum()
    ) == 6807
):

    raise ValueError(
        "PAQ706 differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: PAQ706 matches CDC release range/count."
)


# ============================================================
# 14. CDC CHECK - PAQ605
# ============================================================

check_frequency(
    "PAQ605",
    {
        1: 947,
        2: 5830,
        7: 1,
        9: 1
    },
    2328
)


# ============================================================
# 15. CDC CHECK - PAQ610
# ============================================================

#
# Days vigorous work.
#
# 1-7 = 947
#

s = df["PAQ610"]


if not (
    int(
        s.between(
            1,
            7
        ).sum()
    ) == 947

    and int(
        s.eq(77).sum()
    ) == 0

    and int(
        s.eq(99).sum()
    ) == 0

    and int(
        s.isna().sum()
    ) == 8160
):

    raise ValueError(
        "PAQ610 differs from CDC."
    )


print(
    "PASS: PAQ610 matches CDC."
)


# ============================================================
# 16. CDC CHECK - PAD615
# ============================================================

#
# Minutes vigorous-intensity work.
#
# CDC:
#
# 10-780 = 941
# 9999   = 2
# Missing = 8,164
#

s = df["PAD615"]


if not (
    int(
        s.between(
            10,
            780
        ).sum()
    ) == 941

    and int(
        s.eq(7777).sum()
    ) == 0

    and int(
        s.eq(9999).sum()
    ) == 2

    and int(
        s.isna().sum()
    ) == 8164
):

    raise ValueError(
        "PAD615 differs from CDC."
    )


print(
    "PASS: PAD615 matches CDC."
)


# ============================================================
# 17. CDC CHECK - PAQ620
# ============================================================

check_frequency(
    "PAQ620",
    {
        1: 2065,
        2: 4712,
        7: 1,
        9: 1
    },
    2328
)


# ============================================================
# 18. CDC CHECK - PAQ625
# ============================================================

#
# Days moderate work.
#
# 1-7 = 2,063
# 99  = 2
#

s = df["PAQ625"]


if not (
    int(
        s.between(
            1,
            7
        ).sum()
    ) == 2063

    and int(
        s.eq(77).sum()
    ) == 0

    and int(
        s.eq(99).sum()
    ) == 2

    and int(
        s.isna().sum()
    ) == 7042
):

    raise ValueError(
        "PAQ625 differs from CDC."
    )


print(
    "PASS: PAQ625 matches CDC."
)


# ============================================================
# 19. CDC CHECK - PAD630
# ============================================================

s = df["PAD630"]


if not (
    int(
        s.between(
            10,
            960
        ).sum()
    ) == 2054

    and int(
        s.eq(7777).sum()
    ) == 0

    and int(
        s.eq(9999).sum()
    ) == 5

    and int(
        s.isna().sum()
    ) == 7048
):

    raise ValueError(
        "PAD630 differs from CDC."
    )


print(
    "PASS: PAD630 matches CDC."
)


# ============================================================
# 20. CDC CHECK - PAQ635
# ============================================================

check_frequency(
    "PAQ635",
    {
        1: 2344,
        2: 4434,
        7: 1
    },
    2328
)


# ============================================================
# 21. CDC CHECK - PAQ640
# ============================================================

s = df["PAQ640"]


if not (
    int(
        s.between(
            1,
            7
        ).sum()
    ) == 2343

    and int(
        s.eq(77).sum()
    ) == 0

    and int(
        s.eq(99).sum()
    ) == 1

    and int(
        s.isna().sum()
    ) == 6763
):

    raise ValueError(
        "PAQ640 differs from CDC."
    )


print(
    "PASS: PAQ640 matches CDC."
)


# ============================================================
# 22. CDC CHECK - PAD645
# ============================================================

s = df["PAD645"]


if not (
    int(
        s.between(
            10,
            720
        ).sum()
    ) == 2336

    and int(
        s.eq(7777).sum()
    ) == 0

    and int(
        s.eq(9999).sum()
    ) == 4

    and int(
        s.isna().sum()
    ) == 6767
):

    raise ValueError(
        "PAD645 differs from CDC."
    )


print(
    "PASS: PAD645 matches CDC."
)


# ============================================================
# 23. CDC CHECK - PAQ650
# ============================================================

check_frequency(
    "PAQ650",
    {
        1: 1909,
        2: 4868,
        7: 1,
        9: 1
    },
    2328
)


# ============================================================
# 24. CDC CHECK - PAQ655
# ============================================================

s = df["PAQ655"]


if not (
    int(
        s.between(
            1,
            7
        ).sum()
    ) == 1909

    and int(
        s.eq(77).sum()
    ) == 0

    and int(
        s.eq(99).sum()
    ) == 0

    and int(
        s.isna().sum()
    ) == 7198
):

    raise ValueError(
        "PAQ655 differs from CDC."
    )


print(
    "PASS: PAQ655 matches CDC."
)


# ============================================================
# 25. CDC CHECK - PAD660
# ============================================================

s = df["PAD660"]


if not (
    int(
        s.between(
            10,
            720
        ).sum()
    ) == 1906

    and int(
        s.eq(7777).sum()
    ) == 0

    and int(
        s.eq(9999).sum()
    ) == 0

    and int(
        s.isna().sum()
    ) == 7201
):

    raise ValueError(
        "PAD660 differs from CDC."
    )


print(
    "PASS: PAD660 matches CDC."
)


# ============================================================
# 26. CDC CHECK - PAQ665
# ============================================================

check_frequency(
    "PAQ665",
    {
        1: 2892,
        2: 3886,
        7: 1
    },
    2328
)


# ============================================================
# 27. CDC CHECK - PAQ670
# ============================================================

s = df["PAQ670"]


if not (
    int(
        s.between(
            1,
            7
        ).sum()
    ) == 2891

    and int(
        s.eq(77).sum()
    ) == 0

    and int(
        s.eq(99).sum()
    ) == 1

    and int(
        s.isna().sum()
    ) == 6215
):

    raise ValueError(
        "PAQ670 differs from CDC."
    )


print(
    "PASS: PAQ670 matches CDC."
)


# ============================================================
# 28. CDC CHECK - PAD675
# ============================================================

s = df["PAD675"]


if not (
    int(
        s.between(
            10,
            600
        ).sum()
    ) == 2887

    and int(
        s.eq(7777).sum()
    ) == 0

    and int(
        s.eq(9999).sum()
    ) == 0

    and int(
        s.isna().sum()
    ) == 6220
):

    raise ValueError(
        "PAD675 differs from CDC."
    )


print(
    "PASS: PAD675 matches CDC."
)


# ============================================================
# 29. CDC CHECK - PAD680
# ============================================================

#
# Minutes sedentary activity.
#
# CDC:
#
# 0-1380 = 6,742
# 7777   = 1
# 9999   = 30
# Missing = 2,334
#
# Attached XPT contained one TINY_VALUE that is
# restored to the allowed zero value.
#

s = df["PAD680"]


print("\n--- PAD680 ---")


print(
    "Restored zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            0,
            1380
        ).sum()
    ) == 6742

    and int(
        s.eq(0).sum()
    ) == 1

    and int(
        s.eq(7777).sum()
    ) == 1

    and int(
        s.eq(9999).sum()
    ) == 30

    and int(
        s.isna().sum()
    ) == 2334
):

    raise ValueError(
        "PAD680 differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: PAD680 matches CDC."
)


# ============================================================
# 30. CDC CHECK - PAQ710
# ============================================================

#
# TV/video hours over past 30 days.
#
# CDC:
#
# 0 = Less than 1 hour        1175
# 1 = 1 hour                  1616
# 2 = 2 hours                 2389
# 3 = 3 hours                 1533
# 4 = 4 hours                  884
# 5 = 5 hours or more         1275
# 8 = Does not watch           193
# 77 = Refused                   1
# 99 = Don't know                12
# Missing                       29
#

check_frequency(
    "PAQ710",
    {
        0: 1175,
        1: 1616,
        2: 2389,
        3: 1533,
        4: 884,
        5: 1275,
        8: 193,
        77: 1,
        99: 12
    },
    29
)


print(
    "PASS: PAQ710 has exactly "
    "1,175 validated zeros."
)


# ============================================================
# 31. CDC CHECK - PAQ715
# ============================================================

#
# Computer/game use outside work/school.
#
# CDC:
#
# 0 = Less than 1 hour        2354
# 1 = 1 hour                  1646
# 2 = 2 hours                 1129
# 3 = 3 hours                  562
# 4 = 4 hours                  289
# 5 = 5 hours or more          508
# 8 = Does not use            2586
# 77 = Refused                   1
# 99 = Don't know                 3
# Missing                       29
#

check_frequency(
    "PAQ715",
    {
        0: 2354,
        1: 1646,
        2: 1129,
        3: 562,
        4: 289,
        5: 508,
        8: 2586,
        77: 1,
        99: 3
    },
    29
)


print(
    "PASS: PAQ715 has exactly "
    "2,354 validated zeros."
)


# ============================================================
# 32. CDC CHECK - PAAQUEX
# ============================================================

#
# Questionnaire source flag for weighting:
#
# 1 = SP Interview Questionnaire target
# 2 = MEC CAPI Questionnaire target
#

check_frequency(
    "PAAQUEX",
    {
        1: 8477,
        2: 630
    },
    0
)


# ============================================================
# 33. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# The raw XPT had no ordinary numeric zeros.
#
# Therefore after correction all zeros should occur
# only in the four validated variables.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from expected.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {final_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly matches "
    "the validated corrections."
)


# ============================================================
# 34. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY allowed numeric changes from the
# pandas-decoded XPT are:
#
# PAQ706:
#     55 tiny -> 0
#
# PAD680:
#     1 tiny -> 0
#
# PAQ710:
#     1,175 tiny -> 0
#
# PAQ715:
#     2,354 tiny -> 0
#
# Nothing else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 21 variables match the "
    "expected corrected XPT exactly."
)


# ============================================================
# 35. EXPORT CSV
# ============================================================

#
# IMPORTANT STANDARD RULE:
#
# NO rounding.
#
# NO:
#
#     float_format=
#
# Although PAQ_G contains no genuine decimal values,
# this keeps the same safe export workflow used for
# Dietary, Lab, and Examination files.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 36. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 37. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 21 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 38. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 39. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 40. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 41. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# PAQ_G contains no genuine fractional variables.
#
# Therefore every populated field in THIS FILE
# should serialize without unnecessary ".0".
#
# Examples:
#
#     62161.0 -> 62161
#     60.0    -> 60
#     0.0     -> 0
#     9999.0  -> 9999
#
#
# IMPORTANT:
#
# This global rule is safe ONLY because PAQ_G has no
# genuine decimal variables.
#
# DO NOT apply this global rule to Dietary files,
# survey weights, or genuine continuous Lab/Exam fields.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting "
    "in any PAQ_G variable."
)


# ============================================================
# 42. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        numeric_values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly "
    "matches validated corrections."
)


# ============================================================
# 43. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 44. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 45. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "PAQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "PAQ706 zero restorations:",
    EXPECTED_TINY_COUNTS["PAQ706"]
)


print(
    "PAD680 zero restorations:",
    EXPECTED_TINY_COUNTS["PAD680"]
)


print(
    "PAQ710 zero restorations:",
    EXPECTED_TINY_COUNTS["PAQ710"]
)


print(
    "PAQ715 zero restorations:",
    EXPECTED_TINY_COUNTS["PAQ715"]
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 PAQ_G XPT -> CSV
Rows: 9,107
Columns: 21
PASS: Structure = 9,107 rows x 21 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 3,585

Tiny values by variable:
PAQ715    2354
PAQ710    1175
PAQ706      55
PAD680       1

PASS: Exact 3,585-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
PAQ706: 55 restored to 0
PAD680: 1 restored to 0
PAQ710: 1,175 restored to 0
PAQ715: 2,354 restored to 0

PASS: 3,585 validated zero values restored.

Tiny values remaining after correction: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
PAQ706: 0 fractional observations
PAQ605: 0 fractional observations
PAQ610: 0 fractional observations
PAD615: 0 fractional observations
PAQ620: 0 fractional observations
PAQ625: 0 fractional observations
PAD630: 0 fractional observations
PAQ635: 0 fractional observations
PAQ640: 0 fractional observations
PAD645: 0 fractio